# week4_chunking2 — PDF 레이아웃 파싱

PDF에서 **이미지 / 테이블 / 텍스트**를 분리 추출해 각각 저장한다.

| 종류 | 저장 위치 |
|------|----------|
| 이미지 | `data/images/page_NNN_img_MM.{ext}` |
| 테이블 | `data/tables/page_NNN_table_MM.csv` + `.md` |
| 텍스트 | `data/texts/section_NNN.json` + `sections_all.json` |

In [ ]:
import re
import csv
import json
from pathlib import Path

import fitz  # PyMuPDF

PDF_PATH         = '../data/registration_of_real_estatee_manual.pdf'
DATA_DIR         = Path('../data')
IMG_DIR          = DATA_DIR / 'images'
TBL_DIR          = DATA_DIR / 'tables'
TXT_DIR          = DATA_DIR / 'texts'

IMG_DIR.mkdir(exist_ok=True)
TBL_DIR.mkdir(exist_ok=True)
TXT_DIR.mkdir(exist_ok=True)

TOC_PAGES         = 5
HEADING_FONT_SIZE = 11.0
SECTION_NUM_RE    = re.compile(r'^\d+(\.\d+)*\.$')

doc = fitz.open(PDF_PATH)
print(f'PDF 로드 완료: 총 {len(doc)} 페이지')
print(f'이미지 -> {IMG_DIR.resolve()}')
print(f'테이블 -> {TBL_DIR.resolve()}')
print(f'텍스트 -> {TXT_DIR.resolve()}')

## 1. 이미지 추출

`page.get_images()` 로 PDF에 **내장된 래스터 이미지**를 추출해 원본 포맷(PNG/JPEG 등)으로 저장한다.

In [ ]:
saved_images = []
seen_xrefs   = set()

for page_idx in range(TOC_PAGES, len(doc)):
    page      = doc[page_idx]
    local_idx = 0
    for img_info in page.get_images(full=True):
        xref = img_info[0]
        if xref in seen_xrefs:
            continue
        seen_xrefs.add(xref)
        base = doc.extract_image(xref)
        ext  = base['ext']
        local_idx += 1
        fname = f'page_{page_idx+1:03d}_img_{local_idx:02d}.{ext}'
        (IMG_DIR / fname).write_bytes(base['image'])
        saved_images.append({
            'page': page_idx+1, 'idx': local_idx, 'xref': xref,
            'width': base['width'], 'height': base['height'], 'file': fname,
        })

print(f'이미지 {len(saved_images)}개 저장 완료')
for item in saved_images[:10]:
    print(f"  p.{item['page']:>3}  {item['width']}x{item['height']}  -> {item['file']}")

## 2. 테이블 추출

`page.find_tables()` 로 표를 감지한 뒤 **CSV**(데이터용)와 **Markdown**(사람이 읽기용) 두 포맷으로 저장한다.

In [ ]:
def _clean(cell):
    return str(cell).replace('\n', ' ').strip() if cell else ''

def table_to_markdown(tab):
    rows = tab.extract()
    if not rows:
        return ''
    header = [_clean(c) for c in rows[0]]
    body   = [[_clean(c) for c in row] for row in rows[1:]]
    sep    = '|' + '|'.join([' --- ' for _ in header]) + '|'
    lines  = ['| ' + ' | '.join(header) + ' |', sep]
    lines += ['| ' + ' | '.join(row) + ' |' for row in body]
    return '\n'.join(lines)

saved_tables = []

for page_idx in range(TOC_PAGES, len(doc)):
    page  = doc[page_idx]
    found = page.find_tables()
    for tbl_idx, tbl in enumerate(found.tables):
        rows = tbl.extract()
        if not rows:
            continue
        base = f'page_{page_idx+1:03d}_table_{tbl_idx+1:02d}'
        with open(TBL_DIR / f'{base}.csv', 'w', newline='', encoding='utf-8-sig') as f:
            csv.writer(f).writerows([[_clean(c) for c in row] for row in rows])
        (TBL_DIR / f'{base}.md').write_text(table_to_markdown(tbl), encoding='utf-8')
        saved_tables.append({
            'page': page_idx+1, 'idx': tbl_idx+1,
            'rows': len(rows), 'cols': len(rows[0]),
            'file_csv': f'{base}.csv', 'file_md': f'{base}.md',
        })

print(f'테이블 {len(saved_tables)}개 저장 완료')
for item in saved_tables[:10]:
    print(f"  p.{item['page']:>3}  {item['rows']}행x{item['cols']}열  -> {item['file_csv']}")

## 3. 텍스트 추출

폰트 크기 기반 섹션 분리 + **표 영역 중복 제거** + **이미지 위치 플레이스홀더** 처리.

- 섹션별: `data/texts/section_NNN.json`
- 전체 합본: `data/texts/sections_all.json`

In [ ]:
def rect_overlap(r1, r2, tol=1):
    return not (r1[2]<=r2[0]+tol or r2[2]<=r1[0]+tol
                or r1[3]<=r2[1]+tol or r2[3]<=r1[1]+tol)

def extract_sections_rich(pdf_path, skip_pages=TOC_PAGES):
    d_          = fitz.open(pdf_path)
    sections    = []
    all_hdg     = {}
    current     = {'num':'0','title':'머리말','content':'','start_page':1}
    pending_num = None
    for pi in range(skip_pages, len(d_)):
        page       = d_[pi]
        found      = page.find_tables()
        tbl_bboxes = [tuple(t.bbox) for t in found.tables]
        tbl_map    = {tuple(t.bbox): table_to_markdown(t) for t in found.tables}
        events = sorted(
            [(b['bbox'][1],'block',b) for b in page.get_text('dict')['blocks']] +
            [(tuple(t.bbox)[1],'table',tuple(t.bbox)) for t in found.tables],
            key=lambda e: e[0])
        for _,etype,obj in events:
            if etype=='table':
                current['content'] += '\n' + tbl_map[obj] + '\n'
                continue
            if obj['type']==1:
                current['content'] += '\n[이미지]\n'
                continue
            if any(rect_overlap(obj['bbox'],tb) for tb in tbl_bboxes):
                continue
            for line in obj['lines']:
                for span in line['spans']:
                    text=span['text'].strip(); size=span['size']
                    if not text: continue
                    if size>=HEADING_FONT_SIZE:
                        if SECTION_NUM_RE.match(text):
                            pending_num=text
                        elif pending_num is not None:
                            num_c=pending_num.rstrip('.')
                            all_hdg[num_c]=text
                            if current['content'].strip(): sections.append(current.copy())
                            current={'num':num_c,'title':text,'content':'','start_page':pi+1}
                            pending_num=None
                        else: pending_num=None
                    elif size>10.0: pending_num=None
                    else:
                        pending_num=None
                        current['content']+=text+'\n'
    if current['content'].strip(): sections.append(current)
    return sections, all_hdg

def get_breadcrumb(num, hdg):
    parts=num.split('.')
    return ' > '.join(hdg['.'.join(parts[:i+1])] for i in range(len(parts)) if '.'.join(parts[:i+1]) in hdg)

sections, all_hdg = extract_sections_rich(PDF_PATH)
print(f'추출된 섹션 수           : {len(sections)}')
print(f'전체 헤딩 수 (중간 포함) : {len(all_hdg)}')
print(f'표 포함 섹션 수          : {sum(1 for s in sections if "|" in s["content"])}')
print(f'이미지 플레이스홀더 섹션 : {sum(1 for s in sections if "[이미지]" in s["content"])}')

In [ ]:
all_sections_data = []
for s in sections:
    bc  = get_breadcrumb(s['num'], all_hdg)
    rec = {
        'section_num'  : s['num'],
        'section_title': s['title'],
        'breadcrumb'   : bc,
        'start_page'   : s['start_page'],
        'content'      : s['content'].strip(),
        'has_table'    : '|' in s['content'],
        'has_image_ref': '[이미지]' in s['content'],
    }
    all_sections_data.append(rec)
    safe = s['num'].replace('.', '_')
    (TXT_DIR / f'section_{safe}.json').write_text(
        json.dumps(rec, ensure_ascii=False, indent=2), encoding='utf-8')

all_path = TXT_DIR / 'sections_all.json'
all_path.write_text(json.dumps(all_sections_data, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'텍스트 저장 완료:')
print(f'  섹션별 파일 : {len(all_sections_data)}개  ->  data/texts/section_*.json')
print(f'  전체 합본   : sections_all.json  ({all_path.stat().st_size/1024:.1f} KB)')

## 4. 저장 결과 요약

In [ ]:
from IPython.display import display, HTML

def dir_size(p):
    t = sum(f.stat().st_size for f in p.iterdir() if f.is_file())
    return f'{t/1024:.1f} KB'

rows  = f'<tr><td>이미지</td><td>{len(saved_images)}개</td><td>{dir_size(IMG_DIR)}</td><td><code>data/images/</code></td></tr>'
rows += f'<tr><td>테이블 CSV</td><td>{len(saved_tables)}개</td><td rowspan=2>{dir_size(TBL_DIR)}</td><td><code>data/tables/*.csv</code></td></tr>'
rows += f'<tr><td>테이블 MD</td><td>{len(saved_tables)}개</td><td><code>data/tables/*.md</code></td></tr>'
rows += f'<tr><td>텍스트 섹션별</td><td>{len(all_sections_data)}개</td><td rowspan=2>{dir_size(TXT_DIR)}</td><td><code>data/texts/section_*.json</code></td></tr>'
rows += f'<tr><td>텍스트 합본</td><td>1개</td><td><code>data/texts/sections_all.json</code></td></tr>'

display(HTML(f'''
<style>table{{border-collapse:collapse;font-size:13px}}
th,td{{padding:6px 14px;border:1px solid #ccc;text-align:left}}
th{{background:#2c3e50;color:white}}</style>
<h3>저장 결과</h3>
<table><tr><th>종류</th><th>파일 수</th><th>총 크기</th><th>경로</th></tr>{rows}</table>
'''))

if saved_tables:
    first = saved_tables[0]
    print(f"\n[테이블 샘플] {first['file_md']}")
    print((TBL_DIR / first['file_md']).read_text(encoding='utf-8')[:600])

img_secs = [s for s in all_sections_data if s['has_image_ref']]
if img_secs:
    print(f'\n[이미지 참조 섹션 (처음 5개)]')
    for s in img_secs[:5]:
        print(f"  [{s['section_num']}] {s['breadcrumb'][:55]}  p.{s['start_page']}")